# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brentjoel/FlyRank-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

Many pages are already *visible* in Google (they collect impressions) but very few people click them. A page that is seen thousands of times and almost never clicked is one of the cheapest things a content team can improve: it already ranks, so the fix is often the title, meta description, or how well the page matches what the searcher wanted, not a whole new article. I picked this lane because the starter data shows the problem is large (section 3), because the method can start simple and explainable (compare each page only to pages at a similar position), and because the output is a ranked review list someone can act on today. This is provisional; I can still switch lanes until the end of Week 4.

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brentjoel/FlyRank-1"
REPO_DIR = "FlyRank-1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # walk up from wherever the kernel started until we find the repo root
    here = Path.cwd()
    while not (here / "data" / "raw").is_dir() and here.parent != here:
        here = here.parent
    os.chdir(here)

DATA = Path("data/raw/content_refresh_anonymized.csv")
assert DATA.exists(), "starter CSV not found - are you inside the repo?"
print("Repo root:", Path.cwd().name, "| starter data found")

Repo root: repo | starter data found


## 2. The question: decision, action, cost of a wrong call

**Research question:** *Among pages that are already visible in search, which ones capture far fewer clicks than pages at a similar ranking position, and should be reviewed first?*

- **Unit of analysis:** one content item (page), using its trailing 90-day metrics.
- **Decision it improves:** *which visible pages an SEO/content editor should review first* for title, meta description, snippet structure, or intent match, instead of picking pages by gut feeling or by raw CTR.
- **Who acts and how:** a content editor or SEO specialist takes the top of a ranked list, checks each page against its search intent, and rewrites the title/meta or adjusts the page, or marks it "monitor" if the low CTR has a good reason.
- **Output:** a ranked review queue of pages with a CTR-gap score, reason codes (e.g. *high impressions*, *strong position*, *CTR far below tier*), and a suggested action.
- **Cost of a wrong call:**
  - *False positive* (page flagged but it was fine): a wasted editor hour or two, and a risk of rewriting a title that was actually working.
  - *False negative* (a real under-performer missed): clicks keep being lost on a page that already earned its ranking.
  - Because editor time is the scarce resource, **precision at the top of the list matters most** (precision@K on the first 20–50 pages).
- **Why data/ML can help at all:** CTR depends heavily on position, so raw CTR is misleading: a page at position 15 with 0.2% CTR may be normal, while a page at position 2 with 0.2% may be a problem. Doing this by hand across ~30,000 pages and 32 clients is impossible. A simple position-adjusted baseline (expected CTR per position tier) is the first step; a model only earns its place if it beats that baseline by also using intent, content type, freshness and volume. **This is not "just train a model"**: the core work is the fair comparison (same position tier, enough volume to trust the numbers) and an honest ranked list.

> Frame in one sentence: *For SEO/content editors deciding which visible pages to review first, I will build a ranked CTR-gap queue from the starter data (and later the warehouse), scoring how far each page's CTR sits below pages at a similar position, measured by precision@K on a by-hand review of the top pages. A wrong call costs editor time or missed clicks. A plain raw-CTR sort isn't enough because CTR depends on position and low-volume pages are noisy. I will claim only observed, decision-support results.*

In [2]:
import pandas as pd

df = pd.read_csv(DATA)
print(f"{len(df):,} pages x {df.shape[1]} columns, {df['client_id'].nunique()} clients")

# Gotcha from the data dictionary: avg_position == 0 means "no data", not rank 0.
no_pos = (df["avg_position"] == 0).sum()
print(f"pages with avg_position = 0 (no position data, excluded below): {no_pos:,}")

# Note: ctr is already a x100 percentage (0.76 means 0.76%).

30,000 pages x 44 columns, 32 clients
pages with avg_position = 0 (no position data, excluded below): 1,205


## 3. Quick look at the data (2-3 real numbers)

*Below: the starter CSV, pages with real position data only. Numbers are computed in the code cell, not typed by hand.*

What the numbers show (from the outputs below):

1. **CTR depends strongly on position.** Pooled CTR is about **0.49%** in the top-3 tier, **0.35%** for positions 3–20, **0.16%** at positions 20–50, and **0.04%** beyond 50 (more than a 10x drop from top to bottom). So any fair comparison has to happen *within* a position tier.
2. **Visibility is concentrated.** Pages with at least 1,000 impressions in 90 days are **13,512 pages (45%)**, but they hold **97.5% of all impressions**. That is the pool where a CTR fix can matter.
3. **The opportunity is large.** Among those visible pages, **3,380 (25%)** have a CTR below *half* the median of their own position tier. Together they received **about 30 million impressions (about 20% of the visible pool)** across **20 of the 32 clients**, and **1,149** of the visible pages got **zero clicks** in 90 days.

That is why the lane looks worth the next 7 weeks: the gap is common, it sits on pages that already rank, and it can be measured fairly.

In [3]:
MIN_IMPR = 1_000  # volume floor: below this, CTR is too noisy to judge

pos = df[df["avg_position"] > 0].copy()
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

# Number 1: pooled CTR by position tier (clicks / impressions, in %)
tiers = (pos.groupby("position_tier")
            .agg(pages=("content_id", "size"),
                 min_pos=("avg_position", "min"),
                 max_pos=("avg_position", "max"),
                 impressions=("impressions_90d", "sum"),
                 clicks=("clicks_90d", "sum"))
            .reindex(tier_order))
tiers["pooled_ctr_pct"] = (100 * tiers["clicks"] / tiers["impressions"]).round(3)
print("1) CTR by position tier")
print(tiers.to_string())

# Number 2: how much of the visibility sits on pages with enough volume
visible = pos[pos["impressions_90d"] >= MIN_IMPR].copy()
share = visible["impressions_90d"].sum() / df["impressions_90d"].sum()
print(f"\n2) pages with >= {MIN_IMPR:,} impressions: {len(visible):,} "
      f"({len(visible)/len(df):.0%} of pages) holding {share:.1%} of all impressions")

# Number 3: visible pages whose CTR is below half of their tier's median CTR
visible["tier_median_ctr"] = visible.groupby("position_tier")["ctr"].transform("median")
gap = visible[visible["ctr"] < 0.5 * visible["tier_median_ctr"]]
print(f"\n3) CTR-gap candidates: {len(gap):,} pages ({len(gap)/len(visible):.0%} of visible pages), "
      f"{gap['impressions_90d'].sum():,} impressions "
      f"({gap['impressions_90d'].sum()/visible['impressions_90d'].sum():.0%} of the visible pool), "
      f"spread over {gap['client_id'].nunique()} of {df['client_id'].nunique()} clients")
print(f"   visible pages with zero clicks in 90 days: {(visible['clicks_90d'] == 0).sum():,}")

1) CTR by position tier
               pages  min_pos  max_pos  impressions  clicks  pooled_ctr_pct
position_tier                                                              
top_3           1116      0.1      3.0      7030689   34348           0.489
page_1         11814      3.0     10.0     89575437  313804           0.350
striking        7304     10.0     20.0     22992054   79754           0.347
page_3_5        7242     20.0     50.0     35182261   54499           0.155
deep            1319     50.0    245.0      1228277     508           0.041

2) pages with >= 1,000 impressions: 13,512 (45% of pages) holding 97.5% of all impressions

3) CTR-gap candidates: 3,380 pages (25% of visible pages), 29,971,155 impressions (20% of the visible pool), spread over 20 of 32 clients
   visible pages with zero clicks in 90 days: 1,149


## 4. Careful words: what I can and can't claim

**What my work will be able to say:**
- *Observed:* "In this 90-day window, these pages had a CTR well below other pages at a similar position."
- *Directional:* "Pages with this pattern (e.g. strong position, high impressions, low CTR) look like better review candidates than a random or raw-CTR pick."
- *Decision-support:* "If an editor has time for 20 pages, this list is a reasonable place to start", checked with precision@K on a by-hand review.

**What it will never say:**
- That a low CTR is *caused* by a bad title or meta description. Low CTR can also come from SERP features (answer boxes, ads, AI overviews), brand vs. non-brand queries, or a mix of many queries behind one average position.
- That rewriting a title *will* raise clicks. This is observational data, not an experiment.
- Anything about "how Google ranks pages" or "predicting Google".
- Anything about a specific client or page by name; the data is pseudonymized and I keep it that way.

**Known limits I will carry forward:** `avg_position` is an average over many queries; low-volume pages are noisy (hence the volume floor); `trend_direction` / `trend_pct` are not used as features (label leakage in this dataset); missing values follow `content_type`, so I will add has-data flags instead of filling with zero.

In [4]:
# Sanity checks that back the careful words above
print("max ctr in data (x100 %):", df["ctr"].max(), "-> 100 means 100%: tiny-volume pages, excluded by the volume floor")
print("visible pages with ctr == 100:", int((visible["ctr"] == 100).sum()))
print("avg_position is a 90-day average, range:", pos["avg_position"].min(), "to", pos["avg_position"].max())
print("missing search_volume by content_type (share):")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(2).to_string())

max ctr in data (x100 %): 100.0 -> 100 means 100%: tiny-volume pages, excluded by the volume floor
visible pages with ctr == 100: 0
avg_position is a 90-day average, range: 0.1 to 245.0
missing search_volume by content_type (share):
content_type
comparison article    0.00
feedly article        1.00
keyword article       0.01


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous IDs, never printed)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.